🎧 **Audio introduction**

> *New to this notebook? The two-minute audio guide  walks you through everything before you start.*


<audio controls>
  <source src="https://raw.githubusercontent.com/fair-vid/video-multimodal-pipeline/main/assets/audio/audio_RecordVideo_notebook_tutorial.wav" type="audio/wav">
</audio>


# ╰┈➤🚪**Demo entry point for paper**
*An Agent-Based Digital Twin for Transparent AI in Student Admissions*

## 🎥 Mock Interview Recording Tool

This notebook is the **starting point for a hands-on demonstration** of the agent-based digital twin framework described in our paper about *AI Agent based Digital Twin for Student Admissions*.

It allows you to record a short mock video interview and observe how video data is produced for later analysis by transparent, inspectable AI agents.  
At the **end of this notebook**, you will find links to the **next notebooks** that continue the full end-to-end pipeline, including document understanding, multimodal fusion, and process-level simulation.

> **Note on availability:**  
> The public GitHub repository for the full framework will be added after the anonymous peer-review process is completed.  
> All notebooks in this demo are designed to be executable independently in Google Colaboratory.

---

## What This Notebook Does

This notebook helps you create **video interview recordings** similar to those used in real university admissions workflows.  
The recordings serve as input for studying and testing transparent AI behavior in a controlled setting.

Your recordings can be used to:

- **Observe AI behavior**: see what information AI agents extract from video interviews  
- **Practice interviews**: test how interview answers are captured and analyzed  
- **Learn by inspection**: understand how AI-based admission processes work step by step  
- **Support research**: contribute to reproducible and transparent AI experiments  

---

## Your Privacy Matters

- Video data stays **inside your Colab session**
- Files are saved **only to your Google Drive**, and only if you choose to do so
- No data is uploaded to external servers
- You remain in full control of your recordings at all times

---

## Getting Started

**Run the cells in order (click the ▶️ play button on each cell):**

1. **Step 1** – Select which interview questions you want to answer  
2. **Step 2** – Grant access to your camera and microphone  
3. **Step 3** – Record your answers (one question at a time)  
4. **Step 4** – Save the videos to Google Drive  
   (`colab-output/dream_applicant_application/0_0/video_interviews`)

**Estimated time:** 5–20 minutes, depending on the number of questions selected.

At the **end of the notebook**, follow the provided links to continue exploring the full digital twin workflow.

> **Alternative:** If you prefer not to record yourself, you can use the **[Alternative Mockup Data Notebook](https://colab.research.google.com/drive/1er0wrqpFetVXZdTNmvhWkW0yL-E55jkI?usp=sharing)** to download pre-recorded video interview data instead.



# Step 1: ⁉️ Select interview questions

In [ ]:
#@title 1.1 Select Program & Questions Setup
questions_tsv_url = "https://raw.githubusercontent.com/fair-vid/fair-vid-synthetic-data/main/interview_questionnaires/interview_questions_v1.tsv" #@param {type:"string"}


import ipywidgets as widgets
from IPython.display import display, clear_output
import os
from google.colab import drive
import pandas as pd


os.makedirs("my_video", exist_ok=True)

df_interview_questions = pd.read_csv(questions_tsv_url, sep='\t')


if 'df_interview_questions' not in globals():
    print("⚠️ Error: df_interview_questions not found. Please load your data first.")
else:
    unique_programs = sorted(df_interview_questions['Study_Program'].unique().tolist())

    interview_questions = []  # Store questions and selected indexes
    interview_time = []
    selected_question_indexes = []  # Will store indexes of selected questions
    selected_program = None

    program_dropdown = widgets.Dropdown(  # Create widgets
        options=unique_programs,
        description='Program:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='80%')
    )

    questions_container = widgets.VBox()
    confirm_button = widgets.Button(
        description='Confirm Selection',
        button_style='success',
        icon='check',
        layout=widgets.Layout(width='200px')
    )
    select_all_button = widgets.Button(
        description='Select All',
        button_style='info',
        icon='check-square',
        layout=widgets.Layout(width='150px')
    )
    deselect_all_button = widgets.Button(
        description='Deselect All',
        button_style='warning',
        icon='square',
        layout=widgets.Layout(width='150px')
    )

    output = widgets.Output()
    selection_output = widgets.Output()

    checkboxes = []  # Store checkboxes for manipulation

    def on_program_change(change):
        global checkboxes, interview_questions, interview_time, selected_program
        selected_program = change['new']

        filtered_df = df_interview_questions[  # Filter dataframe
            df_interview_questions['Study_Program'] == selected_program
        ]

        interview_questions = filtered_df['Question'].tolist()  # Store ALL questions and times
        interview_time = filtered_df['Time'].tolist()

        checkboxes = []          # Create checkboxes for each question
        checkbox_widgets = []

        for i, (q, t) in enumerate(zip(interview_questions, interview_time)):
            checkbox = widgets.Checkbox(
                value=False,
                description=f"Q{i+1} ({t}s): {q[:80]}{'...' if len(q) > 80 else ''}",
                style={'description_width': 'initial'},
                layout=widgets.Layout(width='95%')
            )
            checkboxes.append({
                'checkbox': checkbox,
                'index': i  # 0-based index
            })
            checkbox_widgets.append(checkbox)


        with output:  # Update display
            clear_output(wait=True)
            print(f"📚 Program: {selected_program}")
            print(f"❓ Total available questions: {len(interview_questions)}")
            print(f"⏱️  Total time: {sum(interview_time)}s ({sum(interview_time)/60:.1f} min)")
            print("\n✅ Select the questions you want to answer:\n")


        button_row = widgets.HBox([select_all_button, deselect_all_button])  # Update questions container
        questions_container.children = [button_row] + checkbox_widgets + [confirm_button]


        with selection_output:  # Clear previous selection
            clear_output()

    def on_select_all(b):
        for item in checkboxes:
            item['checkbox'].value = True

    def on_deselect_all(b):
        for item in checkboxes:
            item['checkbox'].value = False

    def on_confirm(b):
        global selected_question_indexes

        selected_question_indexes = []
        os.makedirs("my_video/"+selected_program, exist_ok=True)

        for item in checkboxes:
            if item['checkbox'].value:
                selected_question_indexes.append(item['index'])

        with selection_output:
            clear_output(wait=True)
            print("\n" + "🎯 " + "=" * 75)
            print("YOUR SELECTED QUESTIONS")
            print("=" * 78 + "\n")

            if selected_question_indexes:
                total_selected_time = sum(interview_time[i] for i in selected_question_indexes)
                print(f"✅ Number of selected questions: {len(selected_question_indexes)}")
                print(f"⏱️  Total estimated time: {total_selected_time}s ({total_selected_time/60:.1f} min)")
                print(f"📋 Selected indexes: {selected_question_indexes}")
                print("\n" + "-" * 78 + "\n")

                for idx in selected_question_indexes:
                    print(f"{idx+1}. ⏱️  {interview_time[idx]}s")
                    print(f"   {interview_questions[idx]}")
                    print()
            else:
                print("⚠️ No questions selected. Please select at least one question.")

    # Attach event handlers
    program_dropdown.observe(on_program_change, names='value')
    confirm_button.on_click(on_confirm)
    select_all_button.on_click(on_select_all)
    deselect_all_button.on_click(on_deselect_all)

    # Display interface
    print("🎙️ Interview Questions Selector")
    print("=" * 80)
    display(program_dropdown)
    display(output)
    display(questions_container)
    display(selection_output)

    on_program_change({'new': program_dropdown.value})  # Trigger initial display



🎙️ Interview Questions Selector


Dropdown(description='Program:', layout=Layout(width='80%'), options=("Bachelor's degree 1", "Bachelor's degre…

Output()

VBox()

Output()

# Step 2: 📸 Initialize Camera Permission

In [ ]:
#@title Request browser camera/microphone permissions and confirm.
from IPython.display import display, HTML

display(HTML("""
<div style="border: 1px solid #ccc; padding: 20px; border-radius: 8px; max-width: 400px;">
    <h3>1. Grant Camera Access</h3>
    <p>Click the button below. Your browser will ask for permission.</p>
    <button onclick="initCamera()" style="background-color: #007bff; color: white; padding: 10px 20px; border: none; border-radius: 4px; cursor: pointer; font-size: 16px;">
        📸 Initialize Camera
    </button>
    <p id="status" style="margin-top: 10px; font-weight: bold; color: #555;">Waiting for user...</p>
</div>

<script>
async function initCamera() {
    const status = document.getElementById('status');
    status.innerText = "⌛ Requesting access... Please click 'Allow' in the popup.";

    try {
        const stream = await navigator.mediaDevices.getUserMedia({ video: true, audio: true });
        stream.getTracks().forEach(track => track.stop());
        status.innerText = "✅ Camera is ready! You can now run the Interview cell below.";
        status.style.color = "green";
    } catch (err) {
        status.innerText = "❌";
    }
}
</script>
"""))

# Step 3: 🎙️ Run Interview

In [ ]:
#@title This code creates a video interview recorder in Colab. It displays questions one by one, records timed video responses, and saves them as WebM files.

import time
from IPython.display import display, HTML, clear_output
from google.colab.output import eval_js
from base64 import b64decode
import os
import subprocess
import ipywidgets as widgets
import re



# --- 1. Javascript Logic ---
JS_CODE_CAM_CHECK = """
async function triggerPermissions() {
  try {
    const stream = await navigator.mediaDevices.getUserMedia({ video: true, audio: true });
    await new Promise(resolve => setTimeout(resolve, 6000));
    stream.getTracks().forEach(track => track.stop());
    return "OK";
  } catch (err) {
    return "ERROR: " + err.message;
  }
}
"""

JS_CODE_RECORD = """
async function recordVideo(duration) {
  return new Promise(async (resolve) => {
    // 1. Force lower resolution
    const constraints = {
        audio: true,
        video: { width: { ideal: 480 }, height: { ideal: 360 } }
    };

    const stream = await navigator.mediaDevices.getUserMedia(constraints);
    const video = document.getElementById('video_stream');
    video.srcObject = stream;
    await video.play();

    // 2. Limit Bitrate
    const options = { mimeType: 'video/webm; codecs=vp8', bitsPerSecond: 250000 };
    const recorder = new MediaRecorder(stream, options);
    let data = [];

    recorder.ondataavailable = (event) => data.push(event.data);
    recorder.start();



    // Calculate total seconds
    let secondsLeft = Math.floor(duration / 1000);

    // Find the text box
    const timerDiv = document.getElementById('timer_text');

    // Show the first second immediately
    timerDiv.innerHTML = "Recording... " + secondsLeft + " seconds left";

    // Create a loop that runs every 1000 milliseconds (1 second)
    const countdownLoop = setInterval(() => {
        secondsLeft--;
        if (secondsLeft >= 0) {
            timerDiv.innerHTML = "Recording... " + secondsLeft + " seconds left";
        }
    }, 1000);
    // -------------------------






    // Wait for duration
    await new Promise(resolve => setTimeout(resolve, duration));

    recorder.onstop = () => {
      // --- This stops the timer loop ---
      clearInterval(countdownLoop);
      document.getElementById('timer_text').innerHTML = "Done recording. Processing...";
      // ---------------------------
      stream.getTracks().forEach(track => track.stop());
      const blob = new Blob(data, { type: 'video/webm' });
      const reader = new FileReader();
      reader.onloadend = () => {
        resolve(reader.result);
      };
      reader.readAsDataURL(blob);
    };
    recorder.stop();
  });
}
"""

# --- 2. Permission Check Helper ---
def check_permissions():
    """
    Forces the browser to ask for permission before the actual logic starts.
    """
    print("🔒 Requesting camera/microphone access...")
    print("👉 Please click 'Allow' in the browser popup if prompted.")

    try:
        # We inject the checker function and call it immediately
        result = eval_js(JS_CODE_CAM_CHECK + "triggerPermissions()")
        if result == "OK":
            print("✅ Access granted.")
            time.sleep(3)
            return True
        else:
            print(f"❌ Access denied or error: {result}")
            return False
    except Exception as e:
        print(f"❌ Error checking permissions: {e}")
        return False

# --- 3. Function to run the interview ---
def run_interview(button):
    clear_output()

    # --- STEP 1: PRE-CHECK PERMISSIONS ---
    if not check_permissions():
        print("Cannot proceed without camera permissions. Please restart the cell and allow access.")
        time.sleep(5)
        if not check_permissions():
            print("Cannot proceed without camera permissions. Please restart the cell and allow access.")
            return

    # --- STEP 2: START INTERVIEW ---
    clear_output()
    print("--- Welcome to your video interview! ---")
    print("Your answers will be saved directly in the Colab environment.")
    time.sleep(3)

    for i, question in enumerate(interview_questions):
        if i not in selected_question_indexes:
            continue

        duration_ms = interview_time[i] * 1000
        clear_output(wait=True)

        print(f"Question {i+1}/{len(interview_questions)}  ({interview_time[i]} sec)")
        print(f"➡️  {question}\n")

        ### Display video placeholder
        ###display(HTML('<video id="video_stream" width="400" height="300" style="border: 1px solid black; background-color: #333;" autoplay muted></video>'))
        # Display timer and video placeholder
        display(HTML('''
            <div id="timer_text" style="font-size: 20px; font-weight: bold; color: red; margin-bottom: 10px;"></div>
            <video id="video_stream" width="400" height="300" style="border: 1px solid black; background-color: #333;" autoplay muted></video>
        '''))


        # Clean filename
        safe_name = re.sub(r'[^\w\-_\. ]', '', question).replace(' ', '_')
        filename = f"my_video/{selected_program}/{i+1}_{safe_name}.webm"

        try:
            # Countdown
            for k in range(3, 0, -1):
                print(f"\r🟢 Recording in {k}...", end='')
                time.sleep(1)
            print("\r🔴 Recording now!      ", end='') ; time.sleep(1)

            # Call Javascript to record
            base64_data = eval_js(JS_CODE_RECORD + f"recordVideo({duration_ms})")

            print("\r💾 Processing video...   ", end='')

            data_url_prefix, base64_video = base64_data.split(',')
            video_bytes = b64decode(base64_video)

            #os.makedirs(os.path.dirname(filename), exist_ok=True)

            with open(filename, 'wb') as f:
                f.write(video_bytes)

            file_size = os.path.getsize(filename) / 1024
            print(f"\n✅ Saved: {filename} ({file_size:.1f} KB)")
            time.sleep(2)

        except Exception as e:
            print(f"\nAn error occurred: {e}")
            break

    clear_output(wait=True)
    print("🎉 Interview complete! Thank you. 🎉")
    print(f"Files saved in: my_video/{selected_program}/")

# --- 4. Create and display the button ---
record_button = widgets.Button(
    description='🎙️ Start Recording',
    button_style='success',
    layout=widgets.Layout(width='200px', height='50px')
)

record_button.on_click(run_interview)

display(record_button)
print("Click the button above to start your video interview!")

🎉 Interview complete! Thank you. 🎉
Files saved in: my_video/Bachelor's degree 1/


Button(button_style='success', description='🎙️ Start Recording', layout=Layout(height='50px', width='200px'), …

Click the button above to start your video interview!



# Step 4: 🔵🔴🟡🟢 Save to Google Drive


In [ ]:
#@title can run more than 10 min to convert .WebM files to .mp4 files

import os
import subprocess
from google.colab import drive
import ipywidgets as widgets
from IPython.display import display

def convert_and_save():

    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("Google Drive mounted")
    else:
        print("Google Drive already mounted")

    drive_dream_dir = '/content/drive/MyDrive/colab-output/dream_applicant_application/0_0/video_interviews'
    os.makedirs(drive_dream_dir, exist_ok=True)

    # Walk through all subdirectories in 'my_video'
    print("Converting all files to MP4...")

    for root, dirs, files in os.walk('my_video'):
        # Get the relative path from 'my_video'
        rel_path = os.path.relpath(root, 'my_video')

        # Skip if we're at the root 'my_video' directory with no subdirectories
        if rel_path == '.':
            if not dirs:  # No subdirectories exist
                print("⚠️ No subdirectories found in 'my_video'. Please organize files in subdirectories.")
                continue
            else:
                continue  # Skip root level, process only subdirectories

        # Create corresponding directory structure on Drive
        drive_subdir = os.path.join(drive_dream_dir, rel_path)
        os.makedirs(drive_subdir, exist_ok=True)
        print(f"\n📁 Processing directory: {rel_path}")
        print(f"   Saving to: {drive_subdir}")

        # Process all .webm files in current directory
        webm_files = [f for f in files if f.endswith('.webm')]

        if webm_files:
            for webm_file in webm_files:
                source_file = os.path.join(root, webm_file)
                mp4_filename = webm_file.replace('.webm', '.mp4')
                destination_file = os.path.join(drive_subdir, mp4_filename)

                print(f"   Converting: {webm_file} -> {mp4_filename}")

                result = subprocess.run(
                    ['ffmpeg', '-i', source_file, '-c:v', 'libx264', '-c:a', 'aac', destination_file],
                    capture_output=True,
                    text=True
                )

                if result.returncode == 0:
                    print(f"   ✅ Saved: {mp4_filename}")
                else:
                    print(f"   ❌ Error converting {webm_file}")
        else:
            print(f"   No .webm files found in this directory")

    print("\n✅ All conversions complete!")

    os.makedirs('/content/drive/MyDrive/colab-output/dream_applicant_application/0_0/documents_image', exist_ok=True)


# Create button
convert_button = widgets.Button(
    description='Convert & Save Files',
    button_style='success',
    icon='check'
)

def on_button_click(b):
    convert_and_save()

convert_button.on_click(on_button_click)

# Display button
display(convert_button)

Button(button_style='success', description='Convert & Save Files', icon='check', style=ButtonStyle())

Mounted at /content/drive
Google Drive mounted
Converting all files to MP4...

📁 Processing directory: Bachelor's degree 1
   Saving to: /content/drive/MyDrive/colab-output/dream_applicant_application/0_0/video_interviews/Bachelor's degree 1
   Converting: 2_What_extracurricular_activities_have_prepared_you_for_university.webm -> 2_What_extracurricular_activities_have_prepared_you_for_university.mp4
   ✅ Saved: 2_What_extracurricular_activities_have_prepared_you_for_university.mp4
   Converting: 1_Why_did_you_choose_this_specific_major_over_others.webm -> 1_Why_did_you_choose_this_specific_major_over_others.mp4
   ✅ Saved: 1_Why_did_you_choose_this_specific_major_over_others.mp4

✅ All conversions complete!




---



---



---



---




---

## Next 🦶: Parallel Agents in the Digital Twin Demo

In the full admissions process digital twin, multiple AI agents are triggered **in parallel** once application data becomes available.
For example, **video analysis, audio analysis, and document image processing** normally start at the same time after an interview is recorded or documents are uploaded.

For demonstration purposes in Google Colab, we **flatten these parallel execution paths into a sequence of notebooks**.
Each notebook represents one logical agent or processing stage, while preserving the structure of the overall workflow described in the paper.

---

### **Option A:** Continue with Interview Audio Processing

In the next notebook, we extract **audio tracks (MP3)** from the recorded interview videos.
These audio files are later used for:

* speech transcription,
* fluency and speaking-style analysis,
* consistency and scripted-answer checks.

**Audio Extraction Notebook:**
[https://colab.research.google.com/drive/1jTYrU0brHLeMTI65ZzKUDjL8iiCGddPp?usp=sharing](https://colab.research.google.com/drive/1jTYrU0brHLeMTI65ZzKUDjL8iiCGddPp?usp=sharing)

---

### **Option B:** Run Document Image Processing in Parallel

Document understanding can be tested **independently and in parallel** with video-based analysis.
The following notebook implements the OCR-based document processing agent, which extracts text and spatial layout information from scanned diplomas, certificates, and transcripts.

**Document OCR Notebook (Tesseract-based):**
[https://colab.research.google.com/drive/1RfXAZ_5rwYmLdAiHPgm-bDOBrQu2T7Sw?usp=sharing](https://colab.research.google.com/drive/1RfXAZ_5rwYmLdAiHPgm-bDOBrQu2T7Sw?usp=sharing)

This notebook produces inspectable text outputs and word-level bounding boxes that are later reconciled with vision language model interpretations.

---

After completing either path, additional notebooks will guide you through:

* speech transcription and linguistic analysis,
* multimodal evidence fusion,
* process-level replay and simulation within the admissions digital twin.

> **Note:**
> A public GitHub repository containing all notebooks and orchestration code will be released after the anonymous review process is completed.

---






















